# 07 — Evaluation (Corrected)
No silent checkpoint fallback. Missing trained variants are skipped and explicitly reported. Results are smoke-test results until the full evaluation set is run.

In [ ]:
!pip install -q peft transformers datasets pandas
import os,sys,torch,pandas as pd
from datasets import load_dataset
from transformers import AutoModelForCausalLM,AutoTokenizer
from peft import PeftModel
repo=os.path.abspath(os.getcwd()); sys.path.insert(0,repo)
from src.evaluation.metrics import evaluate
BASE='deepseek-ai/deepseek-coder-1.3b-instruct'
he=load_dataset('openai_humaneval',split='test'); mbpp=load_dataset('mbpp',split='test')
print('Benchmarks:',len(he),'HumanEval /',len(mbpp),'MBPP')

In [ ]:
def adapter_path(tag):
    p=f'./checkpoints/{tag}/final'
    return p if os.path.isfile(os.path.join(p,'adapter_config.json')) else None
def load_variant(tag):
    tok=AutoTokenizer.from_pretrained(BASE,trust_remote_code=True)
    model=AutoModelForCausalLM.from_pretrained(BASE,torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,device_map='auto' if torch.cuda.is_available() else None,trust_remote_code=True)
    if tag!='zero_shot':
        p=adapter_path(tag)
        if p is None: return None,tok
        model=PeftModel.from_pretrained(model,p,is_trainable=False)
    model.eval(); return model,tok
rows=[]
for tag,label in [('zero_shot','Zero-Shot'),('sft','SFT'),('ppo','PPO'),('dpo','DPO')]:
    model,tok=load_variant(tag)
    if model is None: print('SKIPPED:',label,'checkpoint missing'); continue
    for k in (1,3,5):
        h=evaluate(model,tok,he.select(range(min(20,len(he)))),K=k,label=label+' HE')
        m=evaluate(model,tok,mbpp.select(range(min(20,len(mbpp)))),K=k,label=label+' MBPP')
        rows.append({'model':label,'K':k,'N_HE':h['total'],'N_MBPP':m['total'],'HE_Pass@1':h['pass_at_1'],'HE_Fix@K':h[f'fix_at_{k}'],'MBPP_Pass@1':m['pass_at_1'],'MBPP_Fix@K':m[f'fix_at_{k}']})
    del model; import gc; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
df=pd.DataFrame(rows); print(df.to_string(index=False)); df.to_csv('evaluation_results_corrected.csv',index=False)
print('These are smoke-test results (N<=20), not final paper results.')

### Required final runs
Use the same fixed benchmark subset, seed, decoding settings, and executable tests for every available checkpoint. Run the full evaluation only after PPO/DPO checkpoints have been independently verified. Do not reuse earlier simulated values.